# 01 — Exploratory Data Analysis

This notebook verifies the generated train, validation, and test split structure, runs the automated data-contract tests, checks missing values, and describes the training targets. Reusable logic lives in `src/eda.py`; tables are saved to `reports/metrics/` and figures to `reports/figures/`.

In [ ]:
from pathlib import Path
import sys

COLAB_ROOT = Path('/content/SRG-Tracker')
PROJECT_ROOT = COLAB_ROOT if (COLAB_ROOT / 'src').is_dir() else (Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent)
SRC_DIR = PROJECT_ROOT / 'src'
DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
METRICS_DIR = PROJECT_ROOT / 'reports' / 'metrics'
FIGURES_DIR = PROJECT_ROOT / 'reports' / 'figures'
if not SRC_DIR.is_dir():
    raise FileNotFoundError(f'Cannot locate src directory from {Path.cwd()}')
sys.path.insert(0, str(PROJECT_ROOT)) if str(PROJECT_ROOT) not in sys.path else None

try:
    import pandas as pd
    from src.eda import create_eda_outputs, split_overview
    from src.preprocessing import load_split
except ImportError as error:
    raise ImportError('Install project dependencies with: pip install -r requirements.txt') from error

In [ ]:
train = load_split('train')
validation = load_split('validation')
test = load_split('test')
split_overview()

In [ ]:
# Run the same contract tests used by the reproducible pipeline.
import unittest

suite = unittest.defaultTestLoader.discover(str(PROJECT_ROOT / 'tests'))
result = unittest.TextTestRunner(verbosity=1).run(suite)
if not result.wasSuccessful():
    raise AssertionError('Automated pipeline tests failed; inspect the output above.')

In [ ]:
missing_values = train.isna().mean().sort_values(ascending=False).rename('missing_rate')
missing_values.to_frame().head(15)

In [ ]:
train[['final_gpa', 'final_course_score', 'course_outcome', 'learning_pace']].describe(include='all').transpose()

In [ ]:
outputs = create_eda_outputs()
print(f"Saved tables to {METRICS_DIR}")
print(f"Saved figures to {FIGURES_DIR}")
outputs['overview']